In [6]:
!pip install qdrant-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.9/389.9 kB 6.3 MB/s eta 0:00:00


In [9]:
import requests
import pandas as pd
import statsmodels.api as sm

# 1. 설정
BASE_URL = "https://estranged-simple-unknowing.ngrok-free.dev"
HEADERS = {"ngrok-skip-browser-warning": "true"}

# 분석 대상 5개 카테고리 컬렉션 명시
target_collections = [
    "cream_products",
    "lotion_products",
    "skintoner_products",
    "mist_products",
    "essence_products"
]

all_extracted_data = []

print("📂 Qdrant 데이터 통합 수집 중...")

for col_name in target_collections:
    try:
        scroll_url = f"{BASE_URL}/collections/{col_name}/points/scroll"
        # 각 컬렉션별로 모든 데이터를 가져오기 위해 limit을 넉넉히 설정 (필요시 조절)
        payload = {"limit": 2000, "with_payload": True}
        res = requests.post(scroll_url, json=payload, headers=HEADERS)

        if res.status_code == 200:
            points = res.json()['result']['points']
            print(f"✅ {col_name:<18} | 로드된 데이터: {len(points):>4}개")

            for p in points:
                pay = p['payload']
                # 실제 DB의 Key값(Q_score, sales_index 등)이 대소문자까지 일치해야 함을 주의하세요!
                all_extracted_data.append({
                    'category': col_name,
                    'Q': pay.get('Q_score'),
                    'E': pay.get('E_score'),
                    'S': pay.get('S_score'),
                    'P': pay.get('P_score'),
                    'y': pay.get('sales_index')
                })
        else:
            print(f"❌ {col_name:<18} | 연결 실패 (상태 코드: {res.status_code})")

    except Exception as e:
        print(f"⚠️ {col_name:<18} | 에러 발생: {e}")

# 2. 데이터 정제
df = pd.DataFrame(all_extracted_data)

# 결측치가 있는 행(NaN) 제거 - 분석 신뢰도를 위해 필수
original_len = len(df)
df = df.dropna(subset=['Q', 'E', 'S', 'P', 'y'])
cleaned_len = len(df)

print("-" * 50)
print(f"📊 최종 분석 가용한 데이터 총계: {cleaned_len}개 (누락 제거: {original_len - cleaned_len}개)")
print("-" * 50)

# 3. 회귀분석 실시 (y = b0 + b1*Q + b2*E + b3*S + 1.5*b4*P)
if cleaned_len > 10:
    # 질문자님의 요청대로 P값에 1.5 가중치 선반영
    df['P_adj'] = 1.5 * df['P']

    # 독립변수(X)와 종속변수(y) 설정
    X = df[['Q', 'E', 'S', 'P_adj']]
    X = sm.add_constant(X) # 상수항(Intercept) 추가
    y = df['y']

    # OLS 모델 학습
    model = sm.OLS(y, X).fit()

    # 결과 출력
    print("\n[ 최종 회귀분석 결과 보고서 ]")
    print(model.summary())

📂 Qdrant 데이터 통합 수집 중...
✅ cream_products     | 로드된 데이터:   92개
✅ lotion_products    | 로드된 데이터:   24개
✅ skintoner_products | 로드된 데이터:   59개
✅ mist_products      | 로드된 데이터:   18개
✅ essence_products   | 로드된 데이터:  185개
--------------------------------------------------
📊 최종 분석 가용한 데이터 총계: 0개 (누락 제거: 378개)
--------------------------------------------------


In [10]:
# 가장 데이터가 많은 essence_products에서 샘플 하나 확인
test_url = f"{BASE_URL}/collections/essence_products/points/scroll"
test_res = requests.post(test_url, json={"limit": 1, "with_payload": True}, headers=HEADERS)

if test_res.status_code == 200:
    sample_payload = test_res.json()['result']['points'][0]['payload']
    print("📋 DB에 실제 저장된 필드명 목록:")
    print(sample_payload.keys())
    print("\n📋 실제 데이터 내용:")
    print(sample_payload)
else:
    print("샘플 데이터를 가져오지 못했습니다.")

📋 DB에 실제 저장된 필드명 목록:
dict_keys(['olive_id', 'master_id', 'search_text', 'olive_name', 'olive_url', 'category', 'original_price', 'sale_price', 'discount_rate', 'volume', 'olive_review_count', 'olive_rating', 'badges', 'olive_total_ml', 'olive_qty', 'olive_is_promo', 'olive_promo_keyword', 'olive_clean_price', 'olive_price_per_ml', 'naver_name', 'naver_price', 'naver_url', 'naver_review_count', 'naver_total_ml', 'naver_qty', 'naver_is_promo', 'naver_promo_keyword', 'naver_clean_price', 'naver_price_per_ml', 'musinsa_name', 'musinsa_price', 'musinsa_url', 'musinsa_review_count', 'musinsa_total_ml', 'musinsa_qty', 'musinsa_is_promo', 'musinsa_promo_keyword', 'musinsa_clean_price', 'musinsa_price_per_ml', 'naver_id', 'musinsa_id', 'musinsa_image_url', 'naver_image_url', 'olive_image_url', 'Q_mass_total', 'Q_pos_product', 'E_mass_total', 'E_pos_product', 'S_mass_total', 'S_pos_product', 'P_score', 'final_recommend_score'])

📋 실제 데이터 내용:
{'olive_id': 'A000000202920', 'master_id': 'PRD_876841

**post-processing : 회귀분석에서 나온 날것의 계수(b)들을 합계가 1(!00%)이 되는 비율로 변환하고, 음수값을 처리함.**
특히 s점수가 마이너스인 점과, 가중치들의 단위가 너무 큰 점을 해결 -> 실제 서비스에 바로 적용 가능한 현실적인 추정 가중치로 변환

In [12]:
import numpy as np

# 1. 회귀 분석에서 나온 원본 계수 (추출된 값 입력)
raw_coeffs = {
    'Q': 7765.8286,
    'E': 885.0202,
    'S': -18086.0943,  # 문제가 된 음수 값
    'P': 8218.7421
}

# 2. 보정 로직 (Business Logic)
# - 음수가 나온 S는 최소 가중치(예: 다른 변수들의 평균의 10%)로 강제 치환하거나 0으로 처리
adjusted_coeffs = {}
for key, val in raw_coeffs.items():
    if key == 'S' and val < 0:
        adjusted_coeffs[key] = 500  # 최소한의 긍정적 영향력 부여 (보정값)
    else:
        adjusted_coeffs[key] = val

# 3. 정규화 (Normalization)
# - 모든 가중치의 합이 1이 되도록 비율로 변환 (추천 공식에 넣기 가장 좋음)
total = sum(adjusted_coeffs.values())
final_weights = {k: v / total for k, v in adjusted_coeffs.items()}

print("--- [최종 추천 시스템용 가중치 비율] ---")
for k, v in final_weights.items():
    print(f"{k} (가중치): {v:.2%}")

# 4. 최종 추천 점수 산출 함수 (예시)
def get_recommendation_score(q, e, s, p):
    score = (final_weights['Q'] * q +
             final_weights['E'] * e +
             final_weights['S'] * s +
             final_weights['P'] * p)
    return score

--- [최종 추천 시스템용 가중치 비율] ---
Q (가중치): 44.71%
E (가중치): 5.10%
S (가중치): 2.88%
P (가중치): 47.32%


회귀결과를 바탕으로 비즈니스 로직 추가 => 최중 가중치 결정 및 스코어링 함수

In [13]:
import pandas as pd

# 1. 회귀분석 결과(coef) 바탕으로 가중치 재설정
# 음수가 나온 S는 최소 비중으로, 비중이 낮았던 E는 보조 비중으로 조정
raw_weights = {
    'Q': 7765.8,
    'E': 885.0,
    'S': 500.0,   # -18086.1 대신 최소 긍정 가중치로 보정
    'P': 8218.7   # 이미 1.5배 가중치가 녹아있는 값
}

# 2. 가중치 정규화 (전체 합이 1이 되도록)
total_weight = sum(raw_weights.values())
norm_weights = {k: v / total_weight for k, v in raw_weights.items()}

def calculate_final_score(row):
    """
    최종 추천 점수(y_hat)를 계산하는 함수
    정규화된 가중치를 사용하여 0~1 사이의 점수를 산출
    """
    # 설계하신 모델: y = b1*Q + b2*E + b3*S + 1.5*b4*P
    # (이미 P_adj를 사용해 b4를 구했으므로 가중치 비율대로 곱함)

    score = (
        row['Q'] * norm_weights['Q'] +    # 'Q_pos_product' 대신 'Q' 사용
        row['E'] * norm_weights['E'] +    # 'E_pos_product' 대신 'E' 사용
        row['S'] * norm_weights['S'] +    # 'S_pos_product' 대신 'S' 사용
        row['P'] * norm_weights['P']     # 'P_score' 대신 'P' 사용
    )
    return score

# 3. 전체 데이터프레임에 적용 (df는 앞서 5개 카테고리를 합친 데이터)
df['final_recommend_score'] = df.apply(calculate_final_score, axis=1)

# 4. 결과 확인: 점수가 높은 상위 5개 상품 추천
top_recommendations = df.sort_values(by='final_recommend_score', ascending=False).head(5)

print("--- [최종 가중치 비율 반영 결과] ---")
for k, v in norm_weights.items():
    print(f"{k} 지수 반영 비중: {v:.2%}")

print("\n--- [추천 시스템 상위 리스트] ---")
print(top_recommendations[['category', 'final_recommend_score']])


--- [최종 가중치 비율 반영 결과] ---
Q 지수 반영 비중: 44.71%
E 지수 반영 비중: 5.10%
S 지수 반영 비중: 2.88%
P 지수 반영 비중: 47.32%

--- [추천 시스템 상위 리스트] ---
Empty DataFrame
Columns: [category, final_recommend_score]
Index: []


**기획 프로모션 상품(olive_is_promo == True) 전용 회귀분석**

In [23]:
# =====================================================================
# [추가 분석] 기획 프로모션 상품(olive_is_promo == True) 전용 회귀분석
# =====================================================================

import requests
import pandas as pd
import statsmodels.api as sm

BASE_URL = "https://estranged-simple-unknowing.ngrok-free.dev"
HEADERS = {"ngrok-skip-browser-warning": "true", "Content-Type": "application/json"}

target_collections = [
    "cream_products", "lotion_products", "skintoner_products",
    "mist_products", "essence_products"
]

promo_extracted_data = []

print("Qdrant에서 [기획 상품] 데이터만 추출 중...")

for col_name in target_collections:
    try:
        scroll_url = f"{BASE_URL}/collections/{col_name}/points/scroll"

        # filter 조건에 olive_is_promo = True 추가
        payload = {
            "limit": 2000,
            "with_payload": True,
            "filter": {
                "must": [
                    {
                        "key": "olive_is_promo",
                        "match": {"value": True}
                    }
                ]
            }
        }
        res = requests.post(scroll_url, json=payload, headers=HEADERS)

        if res.status_code == 200:
            points = res.json().get('result', {}).get('points', [])
            print(f"{col_name:<18} | 기획 상품 {len(points):>3}개 로드")

            for p in points:
                pay = p['payload']
                promo_extracted_data.append({
                    'category': col_name,
                    'Q': pay.get('Q_pos_product'),
                    'E': pay.get('E_pos_product'),
                    'S': pay.get('S_pos_product'),
                    'P': pay.get('P_score'),
                    'y': pay.get('olive_review_count')
                })
        else:
            print(f"{col_name} 연결 실패 ({res.status_code})")

    except Exception as e:
        print(f"⚠️ {col_name} 에러: {e}")

# 데이터 정제
df_promo = pd.DataFrame(promo_extracted_data).dropna()

print("-" * 50)
print(f"최종 분석 가용한 [기획 상품] 총계: {len(df_promo)}개")
print("-" * 50)

# 회귀분석 실시
if len(df_promo) > 10:
    df_promo['P_adj'] = 1.5 * df_promo['P']

    X_promo = df_promo[['Q', 'E', 'S', 'P_adj']]
    X_promo = sm.add_constant(X_promo)
    y_promo = df_promo['y']

    promo_model = sm.OLS(y_promo, X_promo).fit()

    print("\n[ 기획 프로모션 상품 전용 회귀분석 결과 ]")
    print(promo_model.summary())

    # 계수 요약
    p_coeffs = promo_model.params
    print("\n" + "="*50)
    print("📢 기획 상품(Promo) 추천 계수(b값)")
    print(f" 1. 기본 상수(b0): {p_coeffs['const']:.4f}")
    print(f" 2. 품질(Q) 가중치: {p_coeffs['Q']:.4f}")
    print(f" 3. 가치(E) 가중치: {p_coeffs['E']:.4f}")
    print(f" 4. 감성(S) 가중치: {p_coeffs['S']:.4f}")
    print(f" 5. 가성비(P) 가중치: {p_coeffs['P_adj']:.4f}")
    print("="*50)

    # 일반 상품의 계수와 어떻게 다른지 비교해보세요!

Qdrant에서 [기획 상품] 데이터만 추출 중...
cream_products     | 기획 상품  41개 로드
lotion_products    | 기획 상품   3개 로드
skintoner_products | 기획 상품  22개 로드
mist_products      | 기획 상품   6개 로드
essence_products   | 기획 상품  94개 로드
--------------------------------------------------
최종 분석 가용한 [기획 상품] 총계: 166개
--------------------------------------------------

[ 기획 프로모션 상품 전용 회귀분석 결과 ]
                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.028
Model:                            OLS   Adj. R-squared:                  0.004
Method:                 Least Squares   F-statistic:                     1.161
Date:                Wed, 29 Apr 2026   Prob (F-statistic):              0.330
Time:                        05:08:59   Log-Likelihood:                -1827.1
No. Observations:                 166   AIC:                             3664.
Df Residuals:                     161   BIC:                             3680.
Df Mode

In [28]:
import pandas as pd
import statsmodels.api as sm
import numpy as np
import requests # requests 모듈 추가

# =====================================================================
# 데이터 로드 및 P_new 계산 (pdoPAF6AnXgU 셀의 로직 통합)
# =====================================================================
BASE_URL = "https://estranged-simple-unknowing.ngrok-free.dev"
HEADERS = {"ngrok-skip-browser-warning": "true", "Content-Type": "application/json"}

target_collections = [
    "cream_products", "lotion_products", "skintoner_products",
    "mist_products", "essence_products"
]

promo_extracted_data = []

print("📂 Qdrant에서 [기획 상품] 데이터 추출 및 새로운 P점수 계산 중...")

for col_name in target_collections:
    try:
        scroll_url = f"{BASE_URL}/collections/{col_name}/points/scroll"

        payload = {
            "limit": 2000,
            "with_payload": True,
            "filter": {
                "must": [{"key": "olive_is_promo", "match": {"value": True}}]
            }
        }
        res = requests.post(scroll_url, json=payload, headers=HEADERS)

        if res.status_code == 200:
            points = res.json().get('result', {}).get('points', [])
            for p in points:
                pay = p['payload']
                promo_extracted_data.append({
                    'category': col_name,
                    'Q': pay.get('Q_pos_product'),
                    'E': pay.get('E_pos_product'),
                    'S': pay.get('S_pos_product'),
                    'price_score': pay.get('price_score', 0),
                    'discount_score': pay.get('discount_score', 0),
                    'promo_keyword': pay.get('olive_promo_keyword', ''),
                    'is_promo': pay.get('olive_is_promo', False),
                    'y': pay.get('olive_review_count')
                })
        else:
            print(f"❌ {col_name} 연결 실패 ({res.status_code})")

    except Exception as e:
        print(f"⚠️ {col_name} 에러: {e}")

df_promo = pd.DataFrame(promo_extracted_data)

df_promo['promo_score'] = 0.0
df_promo.loc[df_promo['is_promo'] == True, 'promo_score'] += 0.5

keyword_match = df_promo['promo_keyword'].fillna('').str.contains(r'1\+1|증량')
df_promo.loc[keyword_match, 'promo_score'] += 0.2

df_promo['P_new'] = (df_promo['price_score'] * 0.5) + \
                    (df_promo['discount_score'] * 0.3) + \
                    (df_promo['promo_score'] * 0.2)

df_promo = df_promo.dropna(subset=['Q', 'E', 'S', 'P_new', 'y'])

print("--------------------------------------------------")
print(f"📊 최종 분석 가용한 [기획 상품] 총계: {len(df_promo)}개")
print("--------------------------------------------------")

# =====================================================================
# [최종 단계] 통계적 가중치 기반 + 비즈니스 프로모션 가점 반영 추천 시스템
# (이름표 에러 완벽 해결 버전)
# =====================================================================

# 1. 객관적 가중치 산출 (새로운 P_new 점수 사용)
df_total = df_promo.copy()

# 🚨 X_reg에는 반드시 'P_new'가 들어가야 가중치가 안정적으로 산출됩니다!
X_reg = df_total[['Q', 'E', 'S', 'P_new']] # 'P' 대신 'P_new' 사용
X_reg = sm.add_constant(X_reg)
y_reg = df_total['y']

# 모델 학습
base_model = sm.OLS(y_reg, X_reg).fit()
b = base_model.params

print("✅ 단계 1: 객관적 가중치(b) 산출 완료")
# 출력할 때도 b['P_new']로 부릅니다!
print(f"품질(Q): {b['Q']:.2f}, 가치(E): {b['E']:.2f}, 감성(S): {b['S']:.2f}, 가성비(P_new): {b['P_new']:.2f}\n")

# 2. 비즈니스 로직 적용: 프로모션 보너스 점수 계산
df_total['promo_bonus'] = 0.0

# 기획 상품이면 기본 +0.5
df_total.loc[df_total['is_promo'] == True, 'promo_bonus'] += 0.5

# '1+1' 또는 '증량' 키워드 포함 시 추가 +0.2
keyword_mask = df_total['promo_keyword'].fillna('').str.contains(r'1\+1|증량') # SyntaxWarning 수정
df_total.loc[keyword_mask, 'promo_bonus'] += 0.2

# 3. 최종 추천 점수(final_score) 계산
# 계산할 때도 P_new의 가중치(b['P_new'])를 가져와서 사용합니다!
df_total['final_score'] = (
    b['const'] +
    b['Q'] * df_total['Q'] +
    b['E'] * df_total['E'] +
    b['S'] * df_total['S'] +
    b['P_new'] * (df_total['P_new'] + df_total['promo_bonus']) # 'P' 대신 'P_new' 사용
)

# 4. 결과 확인: 상위 5개 상품 추천
top_5 = df_total.sort_values(by='final_score', ascending=False).head(5)

print("📊 [최종 추천 시스템 결과 - 프로모션 가점 반영]")
print("-" * 70)
for i, row in top_5.iterrows():
    promo_tag = "[기획]" if row['is_promo'] else ""
    keyword_tag = f"({row['promo_keyword']})" if row['promo_bonus'] > 0.5 else ""
    print(f"순위 {top_5.index.get_loc(i)+1}위 | {promo_tag}{row['category']} 상품 {keyword_tag}")
    print(f"   ㄴ 최종 추천 점수: {row['final_score']:,.2f} (보너스 적용: +{row['promo_bonus']})")
print("-" * 70)

📂 Qdrant에서 [기획 상품] 데이터 추출 및 새로운 P점수 계산 중...
--------------------------------------------------
📊 최종 분석 가용한 [기획 상품] 총계: 166개
--------------------------------------------------
✅ 단계 1: 객관적 가중치(b) 산출 완료
품질(Q): -36901.98, 가치(E): 27206.12, 감성(S): -29115.98, 가성비(P_new): -7596.84

📊 [최종 추천 시스템 결과 - 프로모션 가점 반영]
----------------------------------------------------------------------
순위 1위 | [기획]essence_products 상품 
   ㄴ 최종 추천 점수: 14,046.01 (보너스 적용: +0.5)
순위 2위 | [기획]essence_products 상품 
   ㄴ 최종 추천 점수: 13,702.12 (보너스 적용: +0.5)
순위 3위 | [기획]essence_products 상품 
   ㄴ 최종 추천 점수: 11,219.04 (보너스 적용: +0.5)
순위 4위 | [기획]essence_products 상품 
   ㄴ 최종 추천 점수: 10,312.26 (보너스 적용: +0.5)
순위 5위 | [기획]essence_products 상품 
   ㄴ 최종 추천 점수: 9,954.96 (보너스 적용: +0.5)
----------------------------------------------------------------------


In [31]:
import requests
import pandas as pd
import statsmodels.api as sm
import numpy as np

BASE_URL = "https://estranged-simple-unknowing.ngrok-free.dev" # (🚨 현재 접속되는 URL로 확인해주세요)
HEADERS = {"ngrok-skip-browser-warning": "true", "Content-Type": "application/json"}

target_collections = [
    "cream_products", "lotion_products", "skintoner_products",
    "mist_products", "essence_products"
]

promo_extracted_data = []

# =====================================================================
# 1. 데이터 추출 (🚨 누락되었던 'P' 점수 완벽 복구!)
# =====================================================================
print("📂 Qdrant에서 [기획 상품] 데이터 추출 중...")

for col_name in target_collections:
    try:
        scroll_url = f"{BASE_URL}/collections/{col_name}/points/scroll"
        payload = {
            "limit": 2000,
            "with_payload": True,
            "filter": {"must": [{"key": "olive_is_promo", "match": {"value": True}}]}
        }
        res = requests.post(scroll_url, json=payload, headers=HEADERS)

        if res.status_code == 200:
            points = res.json().get('result', {}).get('points', [])
            for p in points:
                pay = p['payload']
                promo_extracted_data.append({
                    'category': col_name,
                    'Q': pay.get('Q_pos_product'),
                    'E': pay.get('E_pos_product'),
                    'S': pay.get('S_pos_product'),
                    'P': pay.get('P_score'),                 # 🚨 여기가 복구되었습니다!
                    'promo_keyword': pay.get('olive_promo_keyword', ''),
                    'is_promo': pay.get('olive_is_promo', False),
                    'y': pay.get('olive_review_count')
                })
        else:
            print(f"❌ {col_name} 연결 실패 ({res.status_code})")
    except Exception as e:
        print(f"⚠️ {col_name} 에러: {e}")

# 데이터프레임 변환 및 결측치 제거
df_total = pd.DataFrame(promo_extracted_data)
df_total = df_total.dropna(subset=['Q', 'E', 'S', 'P', 'y'])

print(f"✅ 데이터 로드 완료! (총 {len(df_total)}개)")

if len(df_total) > 10:
    # =====================================================================
    # 2. 객관적 가중치(b) 산출 (순수 P 사용)
    # =====================================================================
    X_reg = df_total[['Q', 'E', 'S', 'P']]
    X_reg = sm.add_constant(X_reg)
    y_reg = df_total['y']

    base_model = sm.OLS(y_reg, X_reg).fit()
    b = base_model.params
    print(f"\n📊 [1단계] 객관적 가중치: Q({b['Q']:,.0f}), E({b['E']:,.0f}), S({b['S']:,.0f}), P({b['P']:,.0f})")

    # =====================================================================
    # 3. 비즈니스 로직: 프로모션 보너스가 반영된 P_final 계산
    # =====================================================================
    df_total['promo_score'] = 0.0
    df_total.loc[df_total['is_promo'] == True, 'promo_score'] += 0.5

    keyword_mask = df_total['promo_keyword'].fillna('').str.contains('1\+1|증량')
    df_total.loc[keyword_mask, 'promo_score'] += 0.2

    # 질문자님 공식 적용: 기존 가성비(80%) + 프로모션 보너스(20%)
    df_total['P_final'] = (df_total['P'] * 0.8) + (df_total['promo_score'] * 0.2)

    # =====================================================================
    # 4. 최종 추천 점수 계산 (건강한 가중치 b['P'] 적용)
    # =====================================================================
    df_total['final_score'] = (
        b['const'] +
        b['Q'] * df_total['Q'] +
        b['E'] * df_total['E'] +
        b['S'] * df_total['S'] +
        b['P'] * df_total['P_final']
    )

    # =====================================================================
    # 5. 결과 출력
    # =====================================================================
    top_5 = df_total.sort_values(by='final_score', ascending=False).head(5)

    print("\n🏆 [최종 추천 시스템 결과 - 프로모션 가점 반영]")
    print("-" * 75)
    for i, row in top_5.iterrows():
        promo_tag = "[기획]" if row['is_promo'] else ""
        keyword_tag = f"({row['promo_keyword']})" if row['promo_score'] > 0.5 else ""
        print(f"순위 {top_5.index.get_loc(i)+1}위 | {promo_tag} {row['category']} 상품 {keyword_tag}")
        print(f"   ㄴ 추천점수: {row['final_score']:,.2f} (순수 P점수: {row['P']:.2f} -> 프로모션 반영 P: {row['P_final']:.2f})")
    print("-" * 75)

else:
    print("🚨 데이터가 너무 적어 분석을 진행할 수 없습니다.")

<>:74: SyntaxWarning: invalid escape sequence '\+'
<>:74: SyntaxWarning: invalid escape sequence '\+'
/tmp/ipykernel_5647/1597236632.py:74: SyntaxWarning: invalid escape sequence '\+'
  keyword_mask = df_total['promo_keyword'].fillna('').str.contains('1\+1|증량')


📂 Qdrant에서 [기획 상품] 데이터 추출 중...
✅ 데이터 로드 완료! (총 166개)

📊 [1단계] 객관적 가중치: Q(-36,225), E(14,199), S(-20,617), P(25,960)

🏆 [최종 추천 시스템 결과 - 프로모션 가점 반영]
---------------------------------------------------------------------------
순위 1위 | [기획] essence_products 상품 
   ㄴ 추천점수: 15,994.40 (순수 P점수: 0.91 -> 프로모션 반영 P: 0.83)
순위 2위 | [기획] essence_products 상품 
   ㄴ 추천점수: 15,387.48 (순수 P점수: 0.87 -> 프로모션 반영 P: 0.79)
순위 3위 | [기획] essence_products 상품 
   ㄴ 추천점수: 13,335.89 (순수 P점수: 0.89 -> 프로모션 반영 P: 0.81)
순위 4위 | [기획] essence_products 상품 
   ㄴ 추천점수: 13,011.67 (순수 P점수: 0.91 -> 프로모션 반영 P: 0.83)
순위 5위 | [기획] essence_products 상품 
   ㄴ 추천점수: 12,194.71 (순수 P점수: 0.91 -> 프로모션 반영 P: 0.83)
---------------------------------------------------------------------------


In [ ]:
import requests
import pandas as pd
import statsmodels.api as sm
import numpy as np

BASE_URL = "https://estranged-simple-unknowing.ngrok-free.dev" # (🚨 현재 접속되는 URL로 확인해주세요)
HEADERS = {"ngrok-skip-browser-warning": "true", "Content-Type": "application/json"}

target_collections = [
    "cream_products", "lotion_products", "skintoner_products",
    "mist_products", "essence_products"
]

promo_extracted_data = []

# =====================================================================
# 1. 데이터 추출 (🚨 누락되었던 'P' 점수 완벽 복구!)
# =====================================================================
print("📂 Qdrant에서 [기획 상품] 데이터 추출 중...")

for col_name in target_collections:
    try:
        scroll_url = f"{BASE_URL}/collections/{col_name}/points/scroll"
        payload = {
            "limit": 2000,
            "with_payload": True,
            "filter": {"must": [{"key": "olive_is_promo", "match": {"value": True}}]}
        }
        res = requests.post(scroll_url, json=payload, headers=HEADERS)

        if res.status_code == 200:
            points = res.json().get('result', {}).get('points', [])
            for p in points:
                pay = p['payload']
                promo_extracted_data.append({
                    'category': col_name,
                    'Q': pay.get('Q_pos_product'),
                    'E': pay.get('E_pos_product'),
                    'S': pay.get('S_pos_product'),
                    'P': pay.get('P_score'),                 # 🚨 여기가 복구되었습니다!
                    'promo_keyword': pay.get('olive_promo_keyword', ''),
                    'is_promo': pay.get('olive_is_promo', False),
                    'y': pay.get('olive_review_count')
                })
        else:
            print(f"❌ {col_name} 연결 실패 ({res.status_code})")
    except Exception as e:
        print(f"⚠️ {col_name} 에러: {e}")

# 데이터프레임 변환 및 결측치 제거
df_total = pd.DataFrame(promo_extracted_data)
df_total = df_total.dropna(subset=['Q', 'E', 'S', 'P', 'y'])

print(f"✅ 데이터 로드 완료! (총 {len(df_total)}개)")

if len(df_total) > 10:
    # =====================================================================
    # 2. 객관적 가중치(b) 산출 (순수 P 사용)
    # =====================================================================
    X_reg = df_total[['Q', 'E', 'S', 'P']]
    X_reg = sm.add_constant(X_reg)
    y_reg = df_total['y']

    base_model = sm.OLS(y_reg, X_reg).fit()
    b = base_model.params
    print(f"\n📊 [1단계] 객관적 가중치: Q({b['Q']:,.0f}), E({b['E']:,.0f}), S({b['S']:,.0f}), P({b['P']:,.0f})")

    # =====================================================================
    # 3. 비즈니스 로직: 프로모션 보너스가 반영된 P_final 계산
    # =====================================================================
    df_total['promo_score'] = 0.0
    df_total.loc[df_total['is_promo'] == True, 'promo_score'] += 0.5

    keyword_mask = df_total['promo_keyword'].fillna('').str.contains(r'1\+1|증량') # Fixed: used raw string for regex
    df_total.loc[keyword_mask, 'promo_score'] += 0.2

    # 질문자님 공식 적용: 기존 가성비(80%) + 프로모션 보너스(20%)
    df_total['P_final'] = (df_total['P'] * 0.8) + (df_total['promo_score'] * 0.2)

    # =====================================================================
    # 4. 최종 추천 점수 계산 (건강한 가중치 b['P'] 적용)
    # =====================================================================
    df_total['final_score'] = (
        b['const'] +
        b['Q'] * df_total['Q'] +
        b['E'] * df_total['E'] +
        b['S'] * df_total['S'] +
        b['P'] * df_total['P_final'] # Fixed: completed the final_score calculation
    )

**추가 코드 (안 쓸 예정)**
기획 상품(is_promo == True)인 경우 무조건 promo_score에 최소 +0.5점을 깔고 시작하게 만듦 ->
그런데 이 회귀분석은 '이미 기획 상품만 모아놓은 데이터(olive_is_promo == True 필터링)'를 대상으로 돌린 것

즉, 분석에 들어간 모든 상품의 P 점수가 일괄적으로 훅 올라가 버렸고, 회귀 모델은 이 '공통적으로 올라간 점수'를 변수의 영향력(b4)으로 보지 않고 "아, 이 집단은 기본 베이스라인(b0) 자체가 엄청 높구나!"라고 착각하여 상수항(b0)으로 다 흡수해버린 것

In [21]:
# =====================================================================
# [추가 분석] 기획 프로모션 상품(olive_is_promo == True) 전용 회귀분석
# (새로운 P점수 공식 적용 버전)
# =====================================================================

import requests
import pandas as pd
import statsmodels.api as sm
import numpy as np

BASE_URL = "https://estranged-simple-unknowing.ngrok-free.dev"
HEADERS = {"ngrok-skip-browser-warning": "true", "Content-Type": "application/json"}

target_collections = [
    "cream_products", "lotion_products", "skintoner_products",
    "mist_products", "essence_products"
]

promo_extracted_data = []

print("📂 Qdrant에서 [기획 상품] 데이터 추출 및 새로운 P점수 계산 중...")

for col_name in target_collections:
    try:
        scroll_url = f"{BASE_URL}/collections/{col_name}/points/scroll"

        # 기획 상품만 필터링
        payload = {
            "limit": 2000,
            "with_payload": True,
            "filter": {
                "must": [{"key": "olive_is_promo", "match": {"value": True}}]
            }
        }
        res = requests.post(scroll_url, json=payload, headers=HEADERS)

        if res.status_code == 200:
            points = res.json().get('result', {}).get('points', [])
            print(f"✅ {col_name:<18} | 기획 상품 {len(points):>3}개 로드")

            for p in points:
                pay = p['payload']
                promo_extracted_data.append({
                    'category': col_name,
                    'Q': pay.get('Q_pos_product'),
                    'E': pay.get('E_pos_product'),
                    'S': pay.get('S_pos_product'),
                    # 🚨 새로운 P계산을 위해 가격, 할인, 키워드 데이터를 함께 가져옵니다.
                    # (DB에 저장된 실제 컬럼명에 맞춰 수정해 주세요)
                    'price_score': pay.get('price_score', 0),
                    'discount_score': pay.get('discount_score', 0),
                    'promo_keyword': pay.get('olive_promo_keyword', ''),
                    'is_promo': pay.get('olive_is_promo', False),
                    'y': pay.get('olive_review_count')
                })
        else:
            print(f"❌ {col_name} 연결 실패 ({res.status_code})")

    except Exception as e:
        print(f"⚠️ {col_name} 에러: {e}")

# 데이터프레임 변환
df_promo = pd.DataFrame(promo_extracted_data)

# =========================================================
# 🌟 제안해주신 새로운 P 점수 공식 적용 (핵심 로직)
# =========================================================
# 1. 프로모션 점수 계산
df_promo['promo_score'] = 0.0
df_promo.loc[df_promo['is_promo'] == True, 'promo_score'] += 0.5

# Fix SyntaxWarning: use raw string for regex
keyword_match = df_promo['promo_keyword'].fillna('').str.contains(r'1\+1|증량')
df_promo.loc[keyword_match, 'promo_score'] += 0.2

# 2. P 점수 조합 (가격 50% + 할인 30% + 프로모션 20%)
# (만약 DB에 price_score가 따로 없고 기존 P_score만 있다면,
# 기존 P_score * 0.8 + promo_score * 0.2 로 응용하실 수 있습니다.)
df_promo['P_new'] = (df_promo['price_score'] * 0.5) + \
                    (df_promo['discount_score'] * 0.3) + \
                    (df_promo['promo_score'] * 0.2)

# 결측치 제거
df_promo = df_promo.dropna(subset=['Q', 'E', 'S', 'P_new', 'y'])

print("-" * 50)
print(f"📊 최종 분석 가용한 [기획 상품] 총계: {len(df_promo)}개")
print("-" * 50)

# 회귀분석 실시
if len(df_promo) > 10:
    # 제안하신 대로 b4 가중치(1.5)는 그대로 유지하고 P값만 대체합니다.
    df_promo['P_adj'] = 1.5 * df_promo['P_new']

    X_promo = df_promo[['Q', 'E', 'S', 'P_adj']]
    X_promo = sm.add_constant(X_promo)
    y_promo = df_promo['y']

    promo_model = sm.OLS(y_promo, X_promo).fit()

    print("\n[ 기획 프로모션 상품 전용 회귀분석 결과 ]")
    print(promo_model.summary())

    p_coeffs = promo_model.params
    print("\n" + "="*50)
    print("📢 기획 상품(Promo) 추천 계수(b값) - 새 공식 적용")
    # Fix SyntaxError: incomplete input
    print(f" 1. 기본 상수(b0): {p_coeffs['const']:.4f}")
    print(f" 2. 품질(Q) 가중치: {p_coeffs['Q']:.4f}")
    print(f" 3. 가치(E) 가중치: {p_coeffs['E']:.4f}")
    print(f" 4. 감성(S) 가중치: {p_coeffs['S']:.4f}")
    print(f" 5. 가성비(P) 가중치: {p_coeffs['P_adj']:.4f}")
    print("="*50)
else:
    print("🚨 기획 상품 데이터가 너무 적어 회귀분석을 진행할 수 없습니다.")

📂 Qdrant에서 [기획 상품] 데이터 추출 및 새로운 P점수 계산 중...
✅ cream_products     | 기획 상품  41개 로드
✅ lotion_products    | 기획 상품   3개 로드
✅ skintoner_products | 기획 상품  22개 로드
✅ mist_products      | 기획 상품   6개 로드
✅ essence_products   | 기획 상품  94개 로드
--------------------------------------------------
📊 최종 분석 가용한 [기획 상품] 총계: 166개
--------------------------------------------------

[ 기획 프로모션 상품 전용 회귀분석 결과 ]
                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.021
Model:                            OLS   Adj. R-squared:                 -0.004
Method:                 Least Squares   F-statistic:                    0.8441
Date:                Wed, 29 Apr 2026   Prob (F-statistic):              0.499
Time:                        05:06:39   Log-Likelihood:                -1827.7
No. Observations:                 166   AIC:                             3665.
Df Residuals:                     161   BIC:                